# Countdown dataset quick stats

This notebook summarizes basic distribution stats to help define difficulty buckets.


In [8]:
# !pip install pyarrow
# !pip install -U pyarrow pandas

In [9]:
import pandas as pd
from pathlib import Path

train_path = Path("train.parquet")
test_path = Path("test.parquet")

train = pd.read_parquet(train_path)
test = pd.read_parquet(test_path)

def _normalize_nums(nums):
    return [int(x) for x in nums]

def add_features(df):
    df = df.copy()
    nums = df["nums"].apply(_normalize_nums)
    df["nums_len"] = nums.apply(len)
    df["nums_min"] = nums.apply(min)
    df["nums_max"] = nums.apply(max)
    df["nums_range"] = df["nums_max"] - df["nums_min"]
    df["nums_sum"] = nums.apply(sum)
    return df

train_f = add_features(train)
test_f = add_features(test)


In [10]:
def group_stats(df):
    g = df.groupby("nums_len").agg(
        rows=("nums_len", "size"),
        target_min=("target", "min"),
        target_max=("target", "max"),
        target_mean=("target", "mean"),
        nums_min=("nums_min", "min"),
        nums_max=("nums_max", "max"),
        nums_range_mean=("nums_range", "mean")
    )
    g["row_ratio"] = (g["rows"] / len(df)).round(4)
    return g.sort_index()

print("Train rows:", len(train_f))
print("Test rows:", len(test_f))

display(train_f.head(3))
display(test_f.head(3))

display(group_stats(train_f))
display(group_stats(test_f))


Train rows: 327680
Test rows: 1024


,target,nums,data_source,prompt,ability,reward_model,extra_info,nums_len,nums_min,nums_max,nums_range,nums_sum,difficulty_score,level,difficulty
0,98,"[44, 19, 35]",countdown,[{'content': 'A conversation between User and ...,math,"{'ground_truth': {'numbers': [44, 19, 35], 'ta...","{'index': 0, 'split': 'train'}",3,19,44,25,98,7.060311,3,easy
1,64,"[63, 95, 96]",countdown,[{'content': 'A conversation between User and ...,math,"{'ground_truth': {'numbers': [63, 95, 96], 'ta...","{'index': 1, 'split': 'train'}",3,63,96,33,254,7.215177,4,medium
2,28,"[95, 11, 56]",countdown,[{'content': 'A conversation between User and ...,math,"{'ground_truth': {'numbers': [95, 11, 56], 'ta...","{'index': 2, 'split': 'train'}",3,11,95,84,162,7.097438,3,easy


,target,nums,data_source,prompt,ability,reward_model,extra_info,nums_len,nums_min,nums_max,nums_range,nums_sum,difficulty_score,level,difficulty
0,36,"[79, 17, 60]",countdown,[{'content': 'A conversation between User and ...,math,"{'ground_truth': {'numbers': [79, 17, 60], 'ta...","{'index': 0, 'split': 'test'}",3,17,79,62,156,7.123805,4,medium
1,56,"[11, 34, 82, 80]",countdown,[{'content': 'A conversation between User and ...,math,"{'ground_truth': {'numbers': [11, 34, 82, 80],...","{'index': 1, 'split': 'test'}",4,11,82,71,207,8.436497,9,hard
2,49,"[51, 4, 60, 35]",countdown,[{'content': 'A conversation between User and ...,math,"{'ground_truth': {'numbers': [51, 4, 60, 35], ...","{'index': 2, 'split': 'test'}",4,4,60,56,150,7.964434,7,medium


,rows,target_min,target_max,target_mean,nums_min,nums_max,nums_range_mean,row_ratio
nums_len,,,,,,,,
3,160876,10,100,54.722283,1,99,47.426813,0.491
4,166804,10,100,55.122269,1,99,59.298734,0.509


,rows,target_min,target_max,target_mean,nums_min,nums_max,nums_range_mean,row_ratio
nums_len,,,,,,,,
3,490,10,100,54.975510,1,99,46.593878,0.4785
4,534,10,100,55.249064,1,99,60.350187,0.5215


In [11]:
def simple_distribution(df, col):
    vc = df[col].value_counts(dropna=False).sort_index()
    out = pd.DataFrame({
        "count": vc,
        "ratio": (vc / len(df)).round(4),
    })
    return out

print("nums_len distribution (train):")
display(simple_distribution(train_f, "nums_len"))

print("nums_len distribution (test):")
display(simple_distribution(test_f, "nums_len"))

print("target describe (train):")
display(train_f["target"].describe())

print("target describe (test):")
display(test_f["target"].describe())


nums_len distribution (train):


,count,ratio
nums_len,,
3,160876,0.491
4,166804,0.509


nums_len distribution (test):


,count,ratio
nums_len,,
3,490,0.4785
4,534,0.5215


target describe (train):


count    327680.000000
mean         54.925894
std          26.253956
min          10.000000
25%          32.000000
50%          55.000000
75%          78.000000
max         100.000000
Name: target, dtype: float64

target describe (test):


count    1024.000000
mean       55.118164
std        26.537579
min        10.000000
25%        32.000000
50%        55.000000
75%        78.000000
max       100.000000
Name: target, dtype: float64

In [12]:
def bucket_by_nums_len(df):
    buckets = {
        "easy": df[df["nums_len"] <= 3],
        "medium": df[(df["nums_len"] == 4) | (df["nums_len"] == 5)],
        "hard": df[df["nums_len"] >= 6],
    }
    rows = []
    for name, sub in buckets.items():
        if len(sub) == 0:
            rows.append((name, 0, 0.0, None, None))
            continue
        rows.append((
            name,
            len(sub),
            round(len(sub) / len(df), 4),
            int(sub["target"].min()),
            int(sub["target"].max()),
        ))
    return pd.DataFrame(rows, columns=["bucket", "rows", "ratio", "target_min", "target_max"])

display(bucket_by_nums_len(train_f))
display(bucket_by_nums_len(test_f))


,bucket,rows,ratio,target_min,target_max
0,easy,160876,0.491,10.0,100.0
1,medium,166804,0.509,10.0,100.0
2,hard,0,0.000,NaN,NaN


,bucket,rows,ratio,target_min,target_max
0,easy,490,0.4785,10.0,100.0
1,medium,534,0.5215,10.0,100.0
2,hard,0,0.0000,NaN,NaN


## Difficulty definition

Current coarse split in this notebook uses only the length of `nums`:
- easy: `nums_len <= 3`
- medium: `nums_len == 4 or 5`
- hard: `nums_len >= 6`

Below we add a finer 10-level score that combines several numeric features and then bucket by deciles.


In [13]:
import numpy as np

def add_difficulty_score(df):
    df = df.copy()
    # Composite score: length dominates, then target magnitude and number spread.
    df["difficulty_score"] = (
        df["nums_len"] * 1.0
        + np.log1p(df["target"].abs()) * 0.6
        + np.log1p(df["nums_range"]) * 0.4
        + (df["nums_max"] >= 75).astype(int) * 0.3
    )
    return df

train_s = add_difficulty_score(train_f)
test_s = add_difficulty_score(test_f)

# Build 10 buckets from train distribution, then apply to test with same bin edges.
train_s["level"] = pd.qcut(train_s["difficulty_score"], 10, labels=False, duplicates="drop")
train_s["level"] = train_s["level"].astype(int) + 1

bins = pd.qcut(train_s["difficulty_score"], 10, duplicates="drop", retbins=True)[1]
test_s["level"] = pd.cut(test_s["difficulty_score"], bins=bins, labels=False, include_lowest=True)
test_s["level"] = test_s["level"].astype(int) + 1

def level_stats(df):
    g = df.groupby("level").agg(
        rows=("level", "size"),
        target_min=("target", "min"),
        target_max=("target", "max"),
        nums_len_min=("nums_len", "min"),
        nums_len_max=("nums_len", "max"),
        score_min=("difficulty_score", "min"),
        score_max=("difficulty_score", "max"),
    ).sort_index()
    g["row_ratio"] = (g["rows"] / len(df)).round(4)
    return g

display(level_stats(train_s))
display(level_stats(test_s))


,rows,target_min,target_max,nums_len_min,nums_len_max,score_min,score_max,row_ratio
level,,,,,,,,
1,32776,10,100,3,4,4.715996,6.544866,0.1000
2,32771,10,100,3,4,6.545055,6.880695,0.1000
3,32772,10,100,3,4,6.880777,7.116729,0.1000
4,32762,10,100,3,4,7.116737,7.326772,0.1000
5,32764,10,100,3,4,7.326786,7.560725,0.1000
6,32763,10,100,3,4,7.560847,7.780732,0.1000
7,32785,15,100,3,4,7.780778,8.052058,0.1001
8,32760,24,100,4,4,8.052133,8.290827,0.1000
9,32766,36,100,4,4,8.290831,8.566018,0.1000


,rows,target_min,target_max,nums_len_min,nums_len_max,score_min,score_max,row_ratio
level,,,,,,,,
1,102,10,96,3,3,5.343703,6.542975,0.0996
2,95,12,91,3,4,6.548496,6.880633,0.0928
3,111,10,97,3,4,6.880827,7.115774,0.1084
4,96,10,100,3,4,7.120625,7.325940,0.0938
5,95,10,100,3,4,7.328972,7.559951,0.0928
6,94,10,100,3,4,7.564476,7.776989,0.0918
7,101,16,99,3,4,7.785242,8.047214,0.0986
8,117,28,100,4,4,8.053474,8.290640,0.1143
9,114,37,100,4,4,8.291966,8.559764,0.1113


In [14]:
# Build a balanced 15360 subset (1536 per level) from train_s
target_per_level = 1536  # 15360 total, divisible by 1024

samples = []
for level, sub in train_s.groupby("level"):
    if len(sub) < target_per_level:
        print(f"Level {level}: only {len(sub)} rows, taking all.")
        pick = sub
    else:
        pick = sub.sample(n=target_per_level, random_state=42)
    samples.append(pick)

balanced = pd.concat(samples, ignore_index=True)
balanced = balanced.sample(frac=1.0, random_state=42).reset_index(drop=True)

out_path = Path("train_15k_balanced.parquet")
tmp_path = out_path.with_suffix(".parquet.tmp")
balanced.to_parquet(tmp_path, index=False)
tmp_path.replace(out_path)

print("balanced rows:", len(balanced))
display(balanced["level"].value_counts().sort_index())

# Verify parquet is readable after write
_verify = pd.read_parquet(out_path)
print("verify rows:", len(_verify))



balanced rows: 15360


level
1     1536
2     1536
3     1536
4     1536
5     1536
6     1536
7     1536
8     1536
9     1536
10    1536
Name: count, dtype: int64

In [15]:
# Build a balanced 30720 subset (3072 per level) from train_s
target_per_level = 3072  # 30720 total, divisible by 1024

samples = []
for level, sub in train_s.groupby("level"):
    if len(sub) < target_per_level:
        print(f"Level {level}: only {len(sub)} rows, taking all.")
        pick = sub
    else:
        pick = sub.sample(n=target_per_level, random_state=42)
    samples.append(pick)

balanced_30k = pd.concat(samples, ignore_index=True)
balanced_30k = balanced_30k.sample(frac=1.0, random_state=42).reset_index(drop=True)

out_path = Path("train_30k_balanced.parquet")
tmp_path = out_path.with_suffix(".parquet.tmp")
balanced_30k.to_parquet(tmp_path, index=False)
tmp_path.replace(out_path)

print("balanced rows:", len(balanced_30k))
display(balanced_30k["level"].value_counts().sort_index())

# Verify parquet is readable after write
_verify = pd.read_parquet(out_path)
print("verify rows:", len(_verify))



balanced rows: 30720


level
1     3072
2     3072
3     3072
4     3072
5     3072
6     3072
7     3072
8     3072
9     3072
10    3072
Name: count, dtype: int64

 target             nums data_source                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     prompt ability                                                                   reward_model                    extra_info  nums_len  nums_min  nums_max  nums_range  nums_sum  difficulty_score  level difficulty
     36     [79, 17, 60]   countdown     [{'content': 'A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer.
User: Using the numbers [79, 17, 60], create an equation that equals 36. You can use basic arithmetic operations (+, -, *, /) and each number can only be used once. Show your work in <think> </think> tags. And return the final answer in <answer> </answer> tags, for example <answer> (1 + 2) / 3 </answer>.
Assistant: Let me solve this step by step.
<think>', 'role': 'user'}]    math     {'ground_truth': {'numbers': [79, 17, 60], 'target': 36}, 'style': 'rule'} {'index': 0, 'split': 'test'}         3        17        79          62       156          7.123805      4     medium
     56 [11, 34, 82, 80]   countdown [{'content': 'A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer.
User: Using the numbers [11, 34, 82, 80], create an equation that equals 56. You can use basic arithmetic operations (+, -, *, /) and each number can only be used once. Show your work in <think> </think> tags. And return the final answer in <answer> </answer> tags, for example <answer> (1 + 2) / 3 </answer>.
Assistant: Let me solve this step by step.
<think>', 'role': 'user'}]    math {'ground_truth': {'numbers': [11, 34, 82, 80], 'target': 56}, 'style': 'rule'} {'index': 1, 'split': 'test'}         4        11        82          71       207          8.436497      9       hard
     49  [51, 4, 60, 35]   countdown  [{'content': 'A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer.
User: Using the numbers [51, 4, 60, 35], create an equation that equals 49. You can use basic arithmetic operations (+, -, *, /) and each number can only be used once. Show your work in <think> </think> tags. And return the final answer in <answer> </answer> tags, for example <answer> (1 + 2) / 3 </answer>.
Assistant: Let me solve this step by step.
<think>', 'role': 'user'}]    math  {'ground_truth': {'numbers': [51, 4, 60, 35], 'target': 49}, 'style': 'rule'} {'index': 2, 'split': 'test'}         4         4        60          56       150          7.964434      7     medium
     33  [34, 98, 1, 96]   countdown  [{'content': 'A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer.
User: Using the numbers [34, 98, 1, 96], create an equation that equals 33. You can use basic arithmetic operations (+, -, *, /) and each number can only be used once. Show your work in <think> </think> tags. And return the final answer in <answer> </answer> tags, for example <answer> (1 + 2) / 3 </answer>.
Assistant: Let me solve this step by step.
<think>', 'role': 'user'}]    math  {'ground_truth': {'numbers': [34, 98, 1, 96], 'target': 33}, 'style': 'rule'} {'index': 3, 'split': 'test'}         4         1        98          97       229          8.249803      8       hard
     29  [46, 9, 49, 56]   countdown  [{'content': 'A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer.
User: Using the numbers [46, 9, 49, 56], create an equation that equals 29. You can use basic arithmetic operations (+, -, *, /) and each number can only be used once. Show your work in <think> </think> tags. And return the final answer in <answer> </answer> tags, for example <answer> (1 + 2) / 3 </answer>.
Assistant: Let me solve this step by step.
<think>', 'role': 'user'}]    math  {'ground_truth': {'numbers': [46, 9, 49, 56], 'target': 29}, 'style': 'rule'} {'index': 4, 'split': 'test'}         4         9        56          47       160          7.589199      6     medium
     41 [63, 46, 21, 15]   countdown [{'content': 'A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer.
User: Using the numbers [63, 46, 21, 15], create an equation that equals 41. You can use basic arithmetic operations (+, -, *, /) and each number can only be used once. Show your work in <think> </think> tags. And return the final answer in <answer> </answer> tags, for example <answer> (1 + 2) / 3 </answer>.
Assistant: Let me solve this step by step.
<think>', 'role': 'user'}]    math {'ground_truth': {'numbers': [63, 46, 21, 15], 'target': 41}, 'style': 'rule'} {'index': 5, 'split': 'test'}         4        15        63          48       145          7.799330      7     medium
     28     [20, 14, 40]   countdown     [{'content': 'A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer.
User: Using the numbers [20, 14, 40], create an equation that equals 28. You can use basic arithmetic operations (+, -, *, /) and each number can only be used once. Show your work in <think> </think> tags. And return the final answer in <answer> </answer> tags, for example <answer> (1 + 2) / 3 </answer>.
Assistant: Let me solve this step by step.
<think>', 'role': 'user'}]    math     {'ground_truth': {'numbers': [20, 14, 40], 'target': 28}, 'style': 'rule'} {'index': 6, 'split': 'test'}         3        14        40          26        74          6.338712      1       easy
     39 [34, 84, 83, 72]   countdown [{'content': 'A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer.
User: Using the numbers [34, 84, 83, 72], create an equation that equals 39. You can use basic arithmetic operations (+, -, *, /) and each number can only be used once. Show your work in <think> </think> tags. And return the final answer in <answer> </answer> tags, for example <answer> (1 + 2) / 3 </answer>.
Assistant: Let me solve this step by step.
<think>', 'role': 'user'}]    math {'ground_truth': {'numbers': [34, 84, 83, 72], 'target': 39}, 'style': 'rule'} {'index': 7, 'split': 'test'}         4        34        84          50       273          8.086058      8       hard
     73     [78, 45, 50]   countdown     [{'content': 'A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer.
User: Using the numbers [78, 45, 50], create an equation that equals 73. You can use basic arithmetic operations (+, -, *, /) and each number can only be used once. Show your work in <think> </think> tags. And return the final answer in <answer> </answer> tags, for example <answer> (1 + 2) / 3 </answer>.
Assistant: Let me solve this step by step.
<think>', 'role': 'user'}]    math     {'ground_truth': {'numbers': [78, 45, 50], 'target': 73}, 'style': 'rule'} {'index': 8, 'split': 'test'}         3        45        78          33       173          7.292983      4     medium
     62  [26, 92, 12, 3]   countdown  [{'content': 'A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer.
User: Using the numbers [26, 92, 12, 3], create an equation that equals 62. You can use basic arithmetic operations (+, -, *, /) and each number can only be used once. Show your work in <think> </think> tags. And return the final answer in <answer> </answer> tags, for example <answer> (1 + 2) / 3 </answer>.
Assistant: Let me solve this step by step.
<think>', 'role': 'user'}]    math  {'ground_truth': {'numbers': [26, 92, 12, 3], 'target': 62}, 'style': 'rule'} {'index': 9, 'split': 'test'}         4         3        92          89       133          8.585805     10       hard
